# ДЗ2. RAG-система (kaggle-соревнование)

## Введение

В архитектурных и строительных бюро инженеры и проектировщики ежедневно работают с нормативными документами (СНиП, СП, ГОСТ, ФЗ и др.). Поиск точного ответа в них может занимать часы.

В этом домашнем задании вы решаете реальный кейс: строите систему, которая по вопросу:
1) находит релевантный фрагмент нормы в базе документов,
2) формирует связный ответ, максимально близкий к эталонному `ideal_answer`.

Цель: собрать end-to-end пайплайн "retrieval в answer".

Доступ к данным - через закрытое Kaggle-соревнование (ссылка в беседе курса).

Вот несколько правил, который помогут нам сделать работу приятнее и продуктивнее:

- Можно использовать любые свободные источники с обязательным указанием ссылки на них. Если в работе вы используете генеративные модели, их указание обязательно. Иначе баллы за работу могут быть снижены. Также следите за оригинальностью: генеративного кода должно быть не более 60% работы.
- Плагиат не допускается. При обнаружении случаев списывания, 0 за работу выставляется всем участникам нарушения, даже если можно установить, кто у кого списал.
- Старайтесь сделать код как можно более оптимальным. В частности, будет штрафоваться использование циклов в тех случаях, когда операцию можно совершить при помощи инструментов библиотек, о которых рассказывалось в курсе.

**Формат сдачи**

Задания сдаются через систему LMS. Посылка должна содержать:

- ноутбук `homework-02-01-Username.ipynb`  
  где `Username` - ваша фамилия латиницей, без пробелов (например, `homework-02-01-Ivanov.ipynb`);
- комментарий к посылке, в котором укажите ваш Kaggle nicknamе (по нему мы будем искать вас в лидерборде соревнования).

> Если вы используете дополнительные файлы (скрипты, сохранённые веса модели и т.п.), то либо:
> - делайте так, чтобы ноутбук мог их скачать сам (например, с Kaggle / Google Drive),  
> - либо приложите их архивом и явно опишите в ноутбуке, как ими пользоваться.  
> В идеале всё должно работать без ручных правок путей.

### Об оценивании

В этом домашнем задании у вас есть два источника баллов:

1. Баллы за ноутбук (за код, воспроизводимость и эксперименты)
2. Баллы за результат на Kaggle (Public leaderboard)

Важно: баллы за соревнование начисляются только при сданном ноутбуке.

**1) Баллы за ноутбук (максимум 8.0 баллов)**

За реализацию пайплайна в этом ноутбуке можно получить **до 8 баллов**.

Минимальная работа - 5.0 баллов.
Вы получаете 5 баллов, если ноутбук:

- ноутбук запускается без ручных правок,
- данные корректно читаются,
- есть retrieval baseline (поиск фрагмента нормы),
- есть генерация ответа (пусть и простая),
- формируется корректный `submission.csv`,
- и вы сдаёте ноутбук в LMS.

Дальше можно добрать баллы:

* **+1.0 балл** - если вы преодолели минимальный бенчмарк на Kaggle (Public score ≥ `B_min`).
* **+1.0 балл** - осмысленные эксперименты (2+ улучшения + таблица результатов + краткие выводы)
* **+1.0 балл** - если вы преодолели сильный бенчмарк на Kaggle (Public score ≥ `B_strong`).

> Пороговые значения **`B_min`** и **`B_strong`** будут указаны на странице Kaggle.

**2) Баллы за соревнование на Kaggle (до 2.0 баллов)**

Дополнительные баллы начисляются по вашему лучшему результату на приватном лидерборде (*Public leaderboard*) и только если у вас сдан воспроизводимый ноутбук.

Баллы за соревнование состоят из двух независимых бонусов:

A. Бонус за попадание в топ-10 (дополнительно):

* Если вы попали в топ-10 по Public leaderboard -> **+1.0 балл**

B. Бонус за попадание в топ-3% (дополнительно):

* Если вы попали в топ-3% по Public leaderboard -> **+1.0 балл**

Таким образом:

* Участник из топ-10 получит **+1**,
* участник из топ-3% получит ещё **+1**,
* участники, которые одновременно в топ-10 и топ-3%, получат **+2**.

> Примечание: топ-3% считается от числа участников, сдавших хотя бы один валидный сабмит. Если 3% - это дробное число, округляем вверх.

**Суммарно за ДЗ2**

* Максимум за ноутбук: 8.0
* Максимум за Kaggle-бонусы: 2.0
* Максимальная оценка за ДЗ2: 10.0


### Требование к воспроизводимости

В конце ноутбука у вас должен быть отдельный блок "Воспроизводимый пайплайн" с одной кодовой ячейкой, которая из исходных данных собирает финальный `submission.csv`в для Kaggle.

Именно эта ячейка должна воспроизводить ваш итоговый результат. Если финальная модель обучалась долго, допускается загружать заранее сохранённые веса, а не обучать модель с нуля.

> Отсюда мы получаем еще одно ограничение: **решение должно помещаться на карту А100**, т.е. воспроизводиться на ресурсах платного colab'а.

## Основная часть работы (5 баллов)

### Сеттинг и данные

Данные доступны на Kaggle, там же есть их полное описание.

Ниже идет код, как скачать данные с kaggle, если вы работаете в colab.

> Разумеется, в рамках данной задачи работать сразу в kaggle было бы оптимальнее. Если вы работаете прямо в Kaggle Notebooks, то и `kaggle.json` не нужен - данные уже доступны во вкладке Data.

In [ ]:
# cкачать данные с Kaggle в Colab
from google.colab import files
uploaded = files.upload()  # выберите kaggle.json

import os
os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

!pip -q install kaggle
!kaggle --version

`kaggle.json` - это файл с вашими учётными данными для Kaggle API (по сути, токен доступа). Он нужен, чтобы из Colab можно было *авторизоваться и скачать данные соревнования командами `kaggle ...`.

**Как получить `kaggle.json`**

1) Зайдите на Kaggle под своим аккаунтом  
2) Откройте Settings / Account
3) Найдите раздел API и нажмите Create New API Token
4) Kaggle скачает файл `kaggle.json` на ваш компьютер - его и нужно загрузить в Colab.

In [ ]:
COMPETITION = "normative-docs-rag"
!kaggle competitions download -c {COMPETITION} -p ./data
!unzip -o ./data/*.zip -d ./data
!ls -la ./data

### Задание 1. Подготовка данных и первичный EDA (1 балл)

**Цель:** убедиться, что данные читаются корректно, и понять базовые свойства выборки: какие вопросы, какие ответы, как устроены документы.

Что нужно сделать:

1. Загрузить три файла соревнования
2. Проверить базовую целостность данных: размеры таблиц и типы столбцов; уникальность `id`; наличие пропусков в ключевых полях (`question`, `ideal_answer`, `text`).
3. Понять, что именно предсказываем: посмотреть `type` в `train` (например, `yesno`, `factoid`) - сколько и каких типов; посмотреть длины.
4. Проверить связку train - documents: убедиться, что `document_id` из `train` действительно встречается в `documents.id`; оценить, сколько уникальных фрагментов норм покрывает обучение (сколько разных `document_id`).

5. Вывести несколько примеров: взять 5-10 строк из `train` и для каждой: показать `question`, показать `full_doc_title` и `text` соответствующего `document_id` из `documents`, показать `ideal_answer`.

**Результат:** вы понимаете, как устроены данные, что такое релевантный фрагмент нормы, и как выглядит целевой эталонный ответ.


In [ ]:
# базовое исследование данных и распределений
# your code here ฅ^•ﻌ•^ฅ

### Задание 2. Сборка датасета для retrieval и answer (1 балл)


**Цель:** собрать удобные структуры данных. Чтобы дальше было удобно строить retrieval и генерацию ответа, полезно привести данные к понятной структуре.

Предлагаем следующие рекомендации:

1. Документы. Желательно иметь таблицу, где:
- ключ - `documents.id`,
- поля - `text`, `full_doc_title`, `document_title`
- Опционально можно добавить поле вроде `doc_text_full` (например, заголовок + текст), если вы считаете, что так retrieval будет работать лучше.

2. Train. Желательно работать с train в виде, где в каждой строке есть:

- `id`, `question`, `type`, `document_id`, `ideal_answer` и (если вам удобно) - присоединенный `text` из `documents` по `document_id`.
- Это не обязательное требование, но такой oracle-контекст удобно использовать для отладки/валидации answering и анализа ошибок.

3. Test. Для test достаточно структуры:

- `id`, `question` (Ответов и `document_id` там нет - всё нужно предсказывать.)

4. Следует подумать и зафиксировать ваш формат контекста для ответа:

- вы будете отвечать только по одному фрагменту нормы (top-1),
- или по топ-k фрагментам,
- или будете собирать контекст из нескольких фрагментов в один контекстный блок.

**Результат:** у вас есть: удобный индекс/таблица документов, удобный train с привязанным текстом нормы, удобный test и понятное решение, в каком виде retrieval будет отдавать контекст для answer.

In [ ]:
# сборка данных
# your code here ヾ(๑╹◡╹)ﾉ

### Задание 3. Базовый baseline (3 балла)

**Цель:** построить первый end-to-end пайплайн, который:

1. по вопросу находит релевантный фрагмент нормы,
2. формирует осмысленный ответ,
3. позволяет получить корректный `submission.csv` для Kaggle.

Что должно быть в baseline:

1. Retrieval baseline (поиск фрагмента нормы):

- реализовать простой способ поиска релевантного фрагмента из `documents` по `question`.
- допустимые варианты (начать лучше с самого простого):

  - TF-IDF поиск по `documents.text`,
  - BM25 поиск,
  - эмбеддинговый поиск (SentenceTransformer).

2. Answer baseline (как строим текст ответа):

- самый простой вариант: вернуть кусок `documents.text` из top-1 найденного фрагмента;
- более аккуратный вариант (предпочтительно): извлечь 1–3 наиболее релевантных предложения из `documents.text` (экстрактивный ответ);
- следите за длиной генерации.

3. Локальная валидация (минимально необходимая):

- выделить часть `train` под валидацию (простое разбиение достаточно);
- сравнить ответы baseline с `ideal_answer`:

  - можно начать с прокси-метрики (например, ROUGE),
  - или посчитать приближенную метрику, близкую к Kaggle (FRIDA/NLI) на небольшой подвыборке.
- **важно:** добавить 2–3 комментария что видно по качеству.

4. Формирование submission:

- сделать файл `submission.csv` точно в формате соревнования:

  - колонки: `id`, `text_answer`,
  - в submission должны присутствовать все `id` из `test.csv`,
  - `id` не менять, заполнять только `text_answer`.

**Результат:** есть полностью рабочий baseline "от вопроса до сабмита", который можно отправить в Kaggle.


In [ ]:
# базовый pipeline
# your code here (づ｡◕‿‿◕｡)づ

## Улучшения baseline'а (дополнительные баллы)

Дальше начинается про "улучшение и осмысленность". Нужно показать, что вы не просто перебираете модели, а понимаете, где узкие места.


### 1. Углублённый EDA (чтобы понять, что улучшать)

Возможные направления (выберите 2–3 и сделайте выводы):

- Длины и структура:

  - распределение длины вопросов,
  - распределение длины `documents.text`,
  - распределение длины `ideal_answer`,
  - связь длины вопроса/контекста с качеством baseline (хотя бы качественно).

- Типы вопросов (`type`) и их особенности:

  - какие типы вопросов встречаются чаще,
  - где baseline чаще плывёт (например, `yesno` vs `factoid`).
  - подсказка: типы вопросов дают вам ключ к валидации IR.

- Документы и покрытие:

  - сколько разных `document_id` в train,
  - есть ли часто встречающиеся нормы,
  - насколько retrieval baseline часто находит правильный `document_id` (top-k recall для train).

- Ошибки retrieval:

  - посмотреть несколько примеров, где top-1 найденный фрагмент явно не тот,
  - понять, что мешает: терминология, длинные фрагменты, редкие слова, таблицы/перечни.

**Результат:** 2-3 наблюдения, которые напрямую переходят в идеи улучшений.

In [ ]:
# углублённый EDA
# your code here ┌(ಠ_ಠ)┘


#### 2. Улучшения retrieval и/или answering (минимум 2 улучшения)

Нужно реализовать **минимум два** осмысленных улучшения относительно baseline. Возможные варианты:

Улучшения retrieval:

- заменить TF-IDF на BM25 (часто сильный базис для нормативных текстов);
- эмбеддинговый поиск по `documents.text`;
- двухэтапный retrieval: добавить rerank по эмбеддингам или кросс-энкодеру.

Улучшения answering:

- не топ-1, а top-k документов: выбрать лучший фрагмент под экстракцию ответа;
- улучшить экстракцию: не первые предложения, а самые релевантные по сходству с вопросом;
- сжатие/переформулировка ответа:

  - строго по найденному контексту,
  - без добавления фактов вне контекста,
  - с контролем длины и без повторов.

> Это только рекомендации - нет гарантий, что они самые оптимальные.


**Важно:** фиксируйте один и тот же протокол валидации, чтобы сравнение было честным.


In [ ]:
# эксперименты с улучшенными признаками / моделями
# your code here ٩(⁎❛ᴗ❛⁎)۶

#### 3. Таблица экспериментов и выводы

В конце экспериментов полезно аккуратно собрать результаты.

Для каждой конфигурации укажите:

- что именно изменили (retrieval? answering? оба?),
- какой скор на вашей валидации (и какая метрика),
- 1–2 предложения "почему стало лучше/хуже".

Пример таблицы:

| Конфигурация | Retrieval                  | Answering                  | Метрика на val |
| ------------ | ------------------------   | -------------------------- | -------------- |
| baseline     | -         | - | 0.XX       |
| улучшение 1  | -         | - | 0.XX       |
| улучшение 2  | -         | - | 0.XX       |
| улучшение 3  | -         | - | 0.XX       |

## Воспроизводимый пайплайн (обязательный блок, если вы участовали в соревновании)

В этом разделе должна быть **одна аккуратная кодовая ячейка**, которая воспроизводит ваш итоговый сабмит.

Эта ячейка должна:

1. загрузить данные;
2. построить нужные индексы/векторизации для retrieval;
3. загрузить/создать финальную модель (если есть);
4. прогнать инференс на `test`;
5. сохранить `submission.csv` в правильном формате (`id`, `text_answer`).

Если финальная модель обучалась долго:

- сохраните веса,
- в этой ячейке загружайте веса, а не обучайте заново,
- укажите, откуда берете веса (ссылка/путь).

**Требование:** если ассистент запустит только эту ячейку (при условии, что все необходимые функции/классы определены выше), на выходе должен появиться корректный `submission.csv`, соответствующий вашему финальному сабмиту на Kaggle.

In [ ]:
# Воспроизводимый пайплайн: формирование финального submission.csv

# your code here (⌐■_■)

## Завершающий блок: выводы

Пожалуйста, в конце ноутбука добавьте небольшой текстовый вывод (5-10 предложений). Можно ориентироваться на такой план:

- какой baseline вы сделали и что он умеет?
- какие улучшения попробовали и почему?
- что реально помогло и что нет?
- как вы валидировались локально?
- короткий комментарий о результате на Kaggle.

Ответ: # your text here (ಠ.ಠ)